In [8]:
import fine as fn
import pyomo.environ as pyomo


# Step 1: Define the Energy System Model
esM = fn.EnergySystemModel(
    locations={"A"},
    commodities={"electricity"},
    commodityUnitsDict={"electricity": "GW"},
    materials={"steel", "copper"},
    materialUnitsDict={"steel": "tons", "copper": "kg"}
)
esM.pyM = pyomo.ConcreteModel()


In [9]:
esM.processedMaterialBalanceLimit = {
    0: {  # Investment Period 0
        "copper": {"A": 10},  # Standort A, Limit für Copper = 100
        "steel": {"A": 5},    # Standort A, Limit für Steel = 50
    }
}


In [10]:
# Step 2: Add a Material Source (Raw Material Supplier)
esM.add(
    fn.Source(
        esM=esM, 
        name="Steel Supply",
        #materialConsumption="steel",
        commodity="electricity",
        hasCapacityVariable=True,
        materialConsumption={"steel": 2, "copper": 0.5},  # Materials required for commissioning
        materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
    )
)


In [11]:
# Step 4: Add a Storage Component that Requires Materials
esM.add(
    fn.Storage(
        esM=esM,
        name="Battery",
        commodity="electricity",
        chargeEfficiency=0.9,
        dischargeEfficiency=0.9,
        materialConsumption={"steel": 2, "copper": 0.5},  # Materials required for commissioning
        materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
    )
)

In [12]:
esM.declareTimeSets(esM.pyM, timeSeriesAggregation=False, segmentation=False)  # Falls die Parameter benötigt werden


In [13]:
print("🔎 Durchsuche componentModelingDict nach Komponenten...")
for name, component in esM.componentModelingDict.items():
    print(f"✅ Rufe declareMaterialComponentVars() für {name} auf...")
    component.declareMaterialComponentVars(esM,esM.pyM)



🔎 Durchsuche componentModelingDict nach Komponenten...
✅ Rufe declareMaterialComponentVars() für SourceSinkModel auf...
🛠️ Definiere Materialvariablen für <fine.sourceSink.SourceSinkModel object at 0x7f067c785eb0>...
✅ Materialvariablen für <fine.sourceSink.SourceSinkModel object at 0x7f067c785eb0> erfolgreich definiert!
✅ Rufe declareMaterialComponentVars() für StorageModel auf...
🛠️ Definiere Materialvariablen für <fine.storage.StorageModel object at 0x7f067c784470>...
✅ Materialvariablen für <fine.storage.StorageModel object at 0x7f067c784470> erfolgreich definiert!


In [14]:

esM.declareMaterialBalanceConstraints(esM.pyM) 


❓ Existiert investSet? True
📊 Inhalt von investSet: [0]
Declaring material balance constraints...
🔍 Prüfe Material copper an Standort A...
  🔹 Komponente: Steel Supply, Typ: <class 'fine.sourceSink.Source'>
  - MaterialConsumption enthält copper? True
  - MaterialRecovery enthält copper? True
  - Standort aktiv? True
✅ Material copper ist an A vorhanden!
🔍 Prüfe Material steel an Standort A...
  🔹 Komponente: Steel Supply, Typ: <class 'fine.sourceSink.Source'>
  - MaterialConsumption enthält steel? True
  - MaterialRecovery enthält steel? True
  - Standort aktiv? True
✅ Material steel ist an A vorhanden!
🔍 Prüfe Materialbalance für (loc=A, mat=copper, ip=0)
💡 Balance Contributions: 0
📏 Material Limit für (loc=A, mat=copper, ip=0): 10
✅ Balance ist trivial (0 ≤ 10), Constraint wird als Feasible gesetzt.
🔍 Prüfe Materialbalance für (loc=A, mat=steel, ip=0)
💡 Balance Contributions: 0
📏 Material Limit für (loc=A, mat=steel, ip=0): 5
✅ Balance ist trivial (0 ≤ 5), Constraint wird als Feasib

In [15]:
esM.optimize()

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.0540 sec)

Declaring sets, variables and constraints for StorageModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(1.4376 sec)

Declaring shared potential constraint...
		(0.0002 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.0898 sec)

		(0.0000 sec)

❓ Existiert investSet? True
📊 Inhalt von investSet: [0]
Declaring material balance constraints...
🔍 Prüfe Material copper an Standort A...
  🔹 Komponente: Steel Supply, Typ: <class 'fine.sourceSink.Source'>
  - MaterialConsumption enthält copper? True
  - MaterialRecovery enthält copper? True
  - Standort aktiv? True
✅ Material copper ist an A vorhanden!
🔍 Prüfe Material steel an Standort A...
  🔹 Komponente: Steel Supply, Typ: <class 'fine.sourceSink.Source'>
  - MaterialConsumption enthält steel? True
  - Mat

In [16]:
esM.getOptimizationSummary("StorageModel", outputLevel=2)

,,,A
Component,Property,Unit,
